This notebook largely follows the approach of ["An NLP-based approach to assessing a company's maturity level in the digital era" (2024) by Romano, Sperlì, and Vignali](https://www.sciencedirect.com/science/article/pii/S0957417424011588)

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
import string

import nltk
nltk.download()
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import PunktSentenceTokenizer, RegexpTokenizer

import os
import sys

# Comment out all but the appropriate option
where_running = "Google Colab"
# where_running = "Local Machine"

if where_running == "Local Machine":
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)
elif where_running == "Google Colab":
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.wrangling.wrangling_tools import is_non_empty

In [52]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [53]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    'Key potential industry 4.0 solutions to address the identified problems/gaps',
    'Recommended Action Plan to utilise Industry 4.0 Technology',
    'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

In [ ]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

In [ ]:
sent_tokenizer = PunktSentenceTokenizer()

sentences_df = responses_df.copy()
sentences_df['Sentences'] = sentences_df.apply(lambda row: sent_tokenizer.tokenize(row['Response']), axis=1)
sentences_df = sentences_df[['Client ID', 'Question', 'Sentences']].explode('Sentences').reset_index().rename(columns={'index' : 'Sentence Number', 'Sentences' : 'Sentence'})
sentences_df['Sentence Number'] = sentences_df.groupby('Sentence Number').cumcount() + 1
sentences_df

# Preprocessing

In [56]:
lemmatizer = WordNetLemmatizer()

# Found initially that some lemmatized stop words slipped through the elimination step, so including these in order to catch both the original and the lemmatized forms.
# For example, "has" and "have" are in the stop words list, but these are lemmatized to "ha" which is not.
lemmatized_stops = set([lemmatizer.lemmatize(word) for word in stopwords.words('english')] + stopwords.words('english'))

In [65]:
# These acronyms and abbreviations will be expanded into their full forms.
acronyms_abbreviations = {
    'I4.0': 'industry 4.0',
    'tech': 'technology',
}

# Here we ensure that "4.0" is kept together as a word token: this should occur often together with "Industry".
regexpr = "4\\.0|[\\w']+"
re_tokenizer = RegexpTokenizer(regexpr)

In [66]:
def preprocess_text(text):
    cleaned_text = text.replace("[REDACTED]", "")
    for short_form, full_form in acronyms_abbreviations.items():
        matches = re.findall(f'({short_form})[\\s|{re.escape(string.punctuation)}]', cleaned_text)
        for match in matches:
            cleaned_text = re.sub(re.escape(match), full_form, cleaned_text, flags=re.IGNORECASE)
    # Skipped removing punctuation as tokenization already does this.
    cleaned_text = cleaned_text.lower()
    tokenized_answer = re_tokenizer.tokenize(cleaned_text)
    lemmatized_answer = [lemmatizer.lemmatize(word) for word in tokenized_answer]
    final_answer = [lemma for lemma in lemmatized_answer if lemma not in lemmatized_stops]
    return " ".join(final_answer)

In [67]:
# responses_df['Cleaned Response'] = responses_df.apply(lambda row: preprocess_text(row['Response']), axis=1)
sentences_df['Cleaned Sentence'] = sentences_df.apply(lambda row: preprocess_text(row['Sentence']), axis=1)

In [68]:
vectorizer = CountVectorizer(analyzer='word', ngram_range=(1,1))
transform_array = vectorizer.fit_transform(sentences_df['Cleaned Sentence'])
feature_names = vectorizer.get_feature_names_out()

In [69]:
features_df = pd.DataFrame({
    "Feature": feature_names,
    "Count": transform_array.sum(axis=0).getA1()
})

In [75]:
features_df.sort_values('Count', ascending=False)[:30]

,Feature,Count
595,business,3307
1197,digital,3129
4005,system,2734
1063,data,2344
2454,management,2000
3110,process,1743
3975,support,1700
805,company,1600
4046,technology,1579
1173,development,1223


Inspection of the 30 most frequent words[1] suggests that this approach will not yield many useful concepts. Instead, we look at bigrams (word pairs) and trigrams (triples) as manual inspection of a sample of responses suggested this may be more fruitful (e.g., "digital strategy" rather than "digital" or "strategy" occuring without the other).

[1] Excluding stop words

In [76]:
vectorizer = CountVectorizer(analyzer='word', ngram_range=(1,3))
transform_array = vectorizer.fit_transform(sentences_df['Cleaned Sentence'])
feature_names = vectorizer.get_feature_names_out()

In [80]:
features_df = pd.DataFrame({
    "Feature": feature_names,
    "Count": transform_array.sum(axis=0).getA1()
})
features_df['N-Gram'] = features_df['Feature'].apply(lambda feature: len(re_tokenizer.tokenize(feature)))

In [81]:
features_df.sort_values('Count', ascending=False)[:30]

,Feature,Count,N-Gram
8708,business,3307,1
20056,digital,3129,1
66366,system,2734,1
17562,data,2344,1
41597,management,2000,1
52066,process,1743,1
65737,support,1700,1
12655,company,1600,1
68215,technology,1579,1
19571,development,1223,1


In [82]:
features_df[features_df['N-Gram'] > 1].sort_values('Count', ascending=False)[:30]

,Feature,Count,N-Gram
20621,digital technology,614,2
20526,digital strategy,499,2
20758,digital transformation,465,2
17722,data collection,390,2
44977,moving forward,380,2
41952,management system,316,2
17910,data management,304,2
9135,business growth,301,2
24678,erp system,265,2
20462,digital skill,260,2


# Expert supervision